# MAPF-GPT 실행 가이드

**논문**: MAPF-GPT: Imitation Learning for Multi-Agent Pathfinding at Scale (AAAI 2025)  
**원본 레포**: https://github.com/CognitiveAISystems/MAPF-GPT

이 노트북은 MAPF-GPT를 Google Colab에서 실행하는 전체 과정을 단계별로 안내합니다.

> **Runtime 설정**: 상단 메뉴 → Runtime → Change runtime type → **T4 GPU** 선택 권장

## Step 1. 환경 설정

In [ ]:
# 의존성 설치 (버전 고정으로 재현성 보장)
!pip install pogema==1.1.0 -q
!pip install torch==2.1.0 -q
!pip install numpy==1.24.3 -q
!pip install gymnasium==0.29.1 -q
!pip install huggingface_hub -q

print("설치 완료")

In [ ]:
# 원본 MAPF-GPT 레포 클론
!git clone https://github.com/CognitiveAISystems/MAPF-GPT.git
%cd MAPF-GPT
!ls

## Step 2. 버전 확인

In [ ]:
import torch
import pogema
import numpy as np

print(f"PyTorch  : {torch.__version__}")
print(f"POGEMA   : {pogema.__version__}")
print(f"NumPy    : {np.__version__}")
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름 : {torch.cuda.get_device_name(0)}")

## Step 3. 사전학습 모델 다운로드 확인

최초 실행 시 Hugging Face에서 모델 가중치를 자동으로 다운로드합니다.  
- 2M 모델: 약 10MB
- 6M 모델: 약 25MB  
- 85M 모델: 약 1GB

In [ ]:
from huggingface_hub import hf_hub_download
import os

# 2M 모델 가중치 미리 다운로드
print("2M 모델 가중치 다운로드 중...")
model_path = hf_hub_download(
    repo_id="aandreychuk/MAPF-GPT",
    filename="2M.pt",
    local_dir="./weights"
)
print(f"다운로드 완료: {model_path}")
print(f"파일 크기: {os.path.getsize(model_path) / 1024 / 1024:.1f} MB")

## Step 4. 기본 실행 — Maze 맵 (16 에이전트)

In [ ]:
# Maze 맵에서 16 에이전트로 실행
!python example.py \
    --model 2M \
    --num_agents 16 \
    --map_name validation-mazes-seed-000 \
    --max_episode_steps 128

## Step 5. 다양한 조건 실험

In [ ]:
# Random 맵 — 16 에이전트
print("=" * 50)
print("[실험 1] Random 맵 / 16 에이전트 / 2M 모델")
print("=" * 50)
!python example.py --model 2M --num_agents 16 --map_name validation-random-seed-000

In [ ]:
# Warehouse 맵 — 32 에이전트
print("=" * 50)
print("[실험 2] Warehouse 맵 / 32 에이전트 / 2M 모델")
print("=" * 50)
!python example.py --model 2M --num_agents 32 --map_name wfi_warehouse

In [ ]:
# Maze 맵 — 모델 크기 비교 (2M vs 6M)
print("=" * 50)
print("[실험 3] Maze 맵 / 32 에이전트 / 2M 모델")
print("=" * 50)
!python example.py --model 2M --num_agents 32 --map_name validation-mazes-seed-000

print("=" * 50)
print("[실험 4] Maze 맵 / 32 에이전트 / 6M 모델")
print("=" * 50)
!python example.py --model 6M --num_agents 32 --map_name validation-mazes-seed-000

## Step 6. 사용 가능한 맵 목록 확인

In [ ]:
!python example.py --show_map_names

## Step 7. 추론 과정 직접 확인

In [ ]:
import sys
sys.path.insert(0, '/content/MAPF-GPT')

import torch
import torch.nn.functional as F

# 모델 직접 로드
device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint = torch.load('./weights/2M.pt', map_location=device)

print("모델 구성 정보:")
if 'model_args' in checkpoint:
    for k, v in checkpoint['model_args'].items():
        print(f"  {k}: {v}")

print(f"\n학습 반복 횟수: {checkpoint.get('iter_num', 'N/A')}")
print(f"검증 손실값: {checkpoint.get('best_val_loss', 'N/A'):.4f}" if 'best_val_loss' in checkpoint else "검증 손실값: N/A")

In [ ]:
# 모델 파라미터 수 계산
from model import GPT, GPTConfig

if 'model_args' in checkpoint:
    gptconf = GPTConfig(**checkpoint['model_args'])
    model = GPT(gptconf)
    model.load_state_dict(checkpoint['model'])
    model.eval()
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"총 파라미터 수: {total_params:,} ({total_params/1e6:.1f}M)")
    print(f"학습 가능 파라미터: {trainable_params:,}")
    print(f"\n모델 구조:")
    print(model)

## Step 8. 성능 측정 — Success Rate 계산

In [ ]:
import pogema
from pogema import pogema_v0, GridConfig
import numpy as np

def run_episode(model, tokenizer_fn, num_agents=16, map_name=None, max_steps=128, seed=42):
    """단일 에피소드 실행 후 성공 여부와 SoC 반환"""
    # 환경 설정
    grid_config = GridConfig(
        num_agents=num_agents,
        seed=seed,
        max_steps=max_steps,
        density=0.3,
        size=17
    )
    env = pogema_v0(grid_config=grid_config)
    observations, info = env.reset()

    done = [False] * num_agents
    steps = 0

    while not all(done) and steps < max_steps:
        actions = []
        for i, obs in enumerate(observations):
            if done[i]:
                actions.append(0)  # 대기
            else:
                # 토큰화 및 추론 (example.py의 로직 따름)
                actions.append(env.action_space.sample())  # 임시: 랜덤 행동
        observations, rewards, terminated, truncated, info = env.step(actions)
        done = [t or tr for t, tr in zip(terminated, truncated)]
        steps += 1

    success = all(terminated)
    return success, steps

print("환경 연결 테스트 완료")
print("실제 MAPF-GPT 추론은 example.py를 통해 실행됩니다.")

## Step 9. 실행 결과 정리

위 실험들의 결과를 정리합니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 논문 Figure 3 재현 (직접 실행한 결과로 채워주세요)
# 아래는 논문 Table 1의 수치를 시각화한 예시입니다.

scenarios = ['Random', 'Mazes', 'Warehouse', 'Cities-tiles', 'Puzzles']

results = {
    'MAPF-GPT-6M (full)': [97.6, 74.6, 94.1, 82.0, 94.0],
    'noGoal':             [95.7, 71.6, 92.8, 88.4, 92.7],
    'noGA':               [97.0, 37.6, 87.7, 79.1, 92.7],
    'noAH':               [95.6, 85.8, 94.8, 82.2, 91.5],
    'noC2G':              [25.8, 15.1, 11.5, 10.2, 52.5],
}

x = range(len(scenarios))
width = 0.15
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (label, values) in enumerate(results.items()):
    offset = (i - 2) * width
    bars = ax.bar([xi + offset for xi in x], values, width, label=label, color=colors[i], alpha=0.85)

ax.set_xlabel('맵 유형', fontsize=12)
ax.set_ylabel('Success Rate (%)', fontsize=12)
ax.set_title('Ablation Study — 입력 정보별 성능 비교 (MAPF-GPT-6M)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(scenarios, fontsize=11)
ax.set_ylim(0, 110)
ax.legend(loc='upper right', fontsize=9)
ax.axhline(y=100, color='gray', linestyle='--', alpha=0.3)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: ablation_study.png")

In [ ]:
# 모델 크기별 성능 비교 시각화
import matplotlib.pyplot as plt
import numpy as np

# 논문 Figure 3 데이터 (Random 맵 기준)
num_agents = [8, 16, 32, 64]

# 논문 수치 기반 근사값
mapf_gpt_2m  = [1.00, 1.00, 0.98, 0.90]
mapf_gpt_6m  = [1.00, 1.00, 0.99, 0.95]
mapf_gpt_85m = [1.00, 1.00, 1.00, 0.98]
dcc          = [1.00, 0.98, 0.85, 0.60]
scrimp       = [1.00, 0.99, 0.92, 0.75]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(num_agents, mapf_gpt_85m, 'b-o',  label='MAPF-GPT-85M', linewidth=2)
ax.plot(num_agents, mapf_gpt_6m,  'b--s', label='MAPF-GPT-6M',  linewidth=2)
ax.plot(num_agents, mapf_gpt_2m,  'b:^',  label='MAPF-GPT-2M',  linewidth=2)
ax.plot(num_agents, scrimp,        'r-x',  label='SCRIMP',        linewidth=2)
ax.plot(num_agents, dcc,           'g-D',  label='DCC',           linewidth=2)

ax.set_xlabel('에이전트 수', fontsize=12)
ax.set_ylabel('Success Rate', fontsize=12)
ax.set_title('Random 맵 — 모델별 성공률 비교 (논문 Figure 3 재현)', fontsize=12)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('success_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장: success_rate_comparison.png")

## 참고 자료

- 원본 논문: https://arxiv.org/abs/2409.00134  
- 원본 코드: https://github.com/CognitiveAISystems/MAPF-GPT  
- 사전학습 모델: https://huggingface.co/aandreychuk/MAPF-GPT